# 07 — D2: Probe de distribución token a token

Este notebook analiza si el *mode collapse* viene de una distribución de generación muy concentrada o si `greedy decoding` está ocultando alternativas cercanas.

La lógica pesada está en:

```bash
scripts/run_d2_token_probe.py
```

Este notebook solo:
1. muestra los comandos a correr;
2. lee los CSVs generados por D2;
3. grafica métricas principales;
4. muestra ejemplos de alta confianza y bajo margen;
5. deja una interpretación preliminar.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mode_collapse_debug"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Comandos D2

Todavía no hace falta correr esto si no están los checkpoints reales.

Cuando estén los checkpoints, el comando principal será:

In [ ]:
cmd_real = f"""
cd "{PROJECT_ROOT}"

python scripts/run_d2_token_probe.py \\
  --checkpoint-root models/blip_finetuned_5k \\
  --indices data/selected_indices.json \\
  --max-images 30 \\
  --device cpu
"""

print(cmd_real)

Para verificar paths sin correr inferencia:

In [ ]:
cmd_dry_run = f"""
cd "{PROJECT_ROOT}"

python scripts/run_d2_token_probe.py \\
  --checkpoint-root models/blip_finetuned_5k \\
  --indices data/selected_indices.json \\
  --max-images 30 \\
  --device cpu \\
  --dry-run
"""

print(cmd_dry_run)

Smoke test opcional con checkpoints debug actuales. No sirve como resultado final, solo para validar que el script funciona.

In [ ]:
cmd_smoke = f"""
cd "{PROJECT_ROOT}"

python scripts/run_d2_token_probe.py \\
  --allow-debug \\
  --indices data/selected_indices.json \\
  --max-images 1 \\
  --device cpu
"""

print(cmd_smoke)

## 2. Carga de resultados

Los archivos esperados son:

```text
outputs/mode_collapse_debug/d2_token_probe_steps.csv
outputs/mode_collapse_debug/d2_image_summary.csv
outputs/mode_collapse_debug/d2_checkpoint_summary.csv
outputs/mode_collapse_debug/d2_high_confidence_examples.csv
outputs/mode_collapse_debug/d2_low_margin_examples.csv
```

In [ ]:
paths = {
    "steps": OUTPUT_DIR / "d2_token_probe_steps.csv",
    "image_summary": OUTPUT_DIR / "d2_image_summary.csv",
    "checkpoint_summary": OUTPUT_DIR / "d2_checkpoint_summary.csv",
    "high_confidence": OUTPUT_DIR / "d2_high_confidence_examples.csv",
    "low_margin": OUTPUT_DIR / "d2_low_margin_examples.csv",
}

for name, path in paths.items():
    print(f"{name:<20}", "OK" if path.exists() else "FALTA", path)

In [ ]:
missing = [name for name, path in paths.items() if not path.exists()]

if missing:
    print("Todavía faltan outputs de D2:", missing)
    print("Esto es normal si todavía no corriste scripts/run_d2_token_probe.py.")
else:
    steps_df = pd.read_csv(paths["steps"])
    image_df = pd.read_csv(paths["image_summary"])
    checkpoint_df = pd.read_csv(paths["checkpoint_summary"])
    high_conf_df = pd.read_csv(paths["high_confidence"])
    low_margin_df = pd.read_csv(paths["low_margin"])

    print("steps_df:", steps_df.shape)
    print("image_df:", image_df.shape)
    print("checkpoint_df:", checkpoint_df.shape)

## 3. Resumen por checkpoint

Métricas clave:

- `mean_p_top1`: confianza promedio del token más probable.
- `mean_p_top2`: probabilidad promedio del segundo token.
- `mean_gap`: diferencia promedio entre top-1 y top-2.
- `pct_steps_top1_gt_090`: porcentaje de pasos donde top-1 supera 0.90.
- `pct_steps_gap_lt_010`: porcentaje de pasos donde top-1 y top-2 están cerca.

In [ ]:
if not missing:
    cols = [
        "checkpoint",
        "n_images",
        "n_steps_total",
        "mean_p_top1",
        "mean_p_top2",
        "mean_gap",
        "mean_entropy",
        "pct_steps_top1_gt_090",
        "pct_steps_top1_gt_095",
        "pct_steps_gap_lt_010",
    ]

    display(checkpoint_df[cols])

## 4. Gráfico principal

Este gráfico permite ver si el modelo se vuelve progresivamente más confiado a través de los checkpoints.

In [ ]:
if not missing:
    ax = checkpoint_df.plot(
        x="checkpoint",
        y=["mean_p_top1", "mean_p_top2", "mean_gap"],
        kind="bar",
        figsize=(9, 4),
    )

    ax.set_title("D2 — Concentración de distribución por checkpoint")
    ax.set_ylabel("Probabilidad / gap")
    ax.set_xlabel("Checkpoint")
    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = OUTPUT_DIR / "d2_checkpoint_distribution_summary.png"
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Figura guardada en:", fig_path)

## 5. Porcentaje de decisiones muy confiadas o ambiguas

In [ ]:
if not missing:
    ax = checkpoint_df.plot(
        x="checkpoint",
        y=["pct_steps_top1_gt_090", "pct_steps_top1_gt_095", "pct_steps_gap_lt_010"],
        kind="bar",
        figsize=(9, 4),
    )

    ax.set_title("D2 — Pasos muy confiados vs. pasos ambiguos")
    ax.set_ylabel("Proporción de tokens")
    ax.set_xlabel("Checkpoint")
    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = OUTPUT_DIR / "d2_confident_vs_ambiguous_steps.png"
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Figura guardada en:", fig_path)

## 6. Ejemplos de alta confianza

Estos casos sugieren que el modelo no está dudando: el token top-1 domina claramente.

In [ ]:
if not missing:
    cols = [
        "checkpoint",
        "idx",
        "n_steps",
        "mean_p_top1",
        "mean_p_top2",
        "mean_gap",
        "mean_entropy",
        "caption",
        "reference",
    ]

    display(high_conf_df[cols].head(10))

## 7. Ejemplos de bajo margen

Estos casos sugieren que `greedy decoding` podría estar ocultando alternativas razonables.

In [ ]:
if not missing:
    cols = [
        "checkpoint",
        "idx",
        "n_steps",
        "mean_p_top1",
        "mean_p_top2",
        "mean_gap",
        "mean_entropy",
        "caption",
        "reference",
    ]

    display(low_margin_df[cols].head(10))

## 8. Inspección token a token

Permite mirar una imagen concreta y ver, paso por paso, qué token ganó y cuál era la segunda alternativa.

In [ ]:
if not missing:
    checkpoint_to_inspect = checkpoint_df["checkpoint"].iloc[-1]
    idx_to_inspect = image_df[image_df["checkpoint"] == checkpoint_to_inspect]["idx"].iloc[0]

    print("checkpoint_to_inspect:", checkpoint_to_inspect)
    print("idx_to_inspect:", idx_to_inspect)

    token_view = steps_df[
        (steps_df["checkpoint"] == checkpoint_to_inspect)
        & (steps_df["idx"] == idx_to_inspect)
    ][[
        "step",
        "generated_token",
        "generated_text_piece",
        "p_generated",
        "top1_token",
        "top1_text_piece",
        "p_top1",
        "top2_token",
        "top2_text_piece",
        "p_top2",
        "gap_top1_top2",
        "entropy",
    ]]

    display(token_view)

## 9. Interpretación automática preliminar

Esta celda no reemplaza el análisis humano, pero ayuda a clasificar el escenario.

In [ ]:
if not missing:
    for _, row in checkpoint_df.iterrows():
        checkpoint = row["checkpoint"]
        mean_p_top1 = row["mean_p_top1"]
        mean_gap = row["mean_gap"]
        pct_high = row["pct_steps_top1_gt_090"]
        pct_amb = row["pct_steps_gap_lt_010"]

        print("=" * 100)
        print("Checkpoint:", checkpoint)
        print(f"mean_p_top1: {mean_p_top1:.3f}")
        print(f"mean_gap: {mean_gap:.3f}")
        print(f"pct_steps_top1_gt_090: {pct_high:.3f}")
        print(f"pct_steps_gap_lt_010: {pct_amb:.3f}")
        print()

        if mean_p_top1 >= 0.90 and mean_gap >= 0.50:
            print("Lectura preliminar: distribución muy concentrada.")
            print("Interpretación: greedy no parece ser el único problema; el modelo está muy confiado.")
        elif mean_gap <= 0.10 or pct_amb >= 0.40:
            print("Lectura preliminar: hay muchas alternativas cercanas.")
            print("Interpretación: sampling / nucleus / diverse decoding podría mejorar diversidad.")
        else:
            print("Lectura preliminar: escenario mixto.")
            print("Interpretación: conviene mirar ejemplos cualitativos y comparar contra D1.")

        print()

## 10. Conclusión para informe

Completar después de correr D2 real:

- Si `mean_p_top1` y `mean_gap` son altos, el collapse parece estar internalizado en la distribución del modelo.
- Si `mean_gap` es bajo o hay muchos pasos con `gap < 0.10`, greedy decoding puede estar amplificando diferencias pequeñas.
- Si la confianza aumenta de `epoch_1` a `best`, eso sugiere que el entrenamiento vuelve al modelo progresivamente más rígido.